# Math 300, Fall 2026 — Project 1
## Root-Finding and Fixed-Point Iteration

**Name:** _your name here_

Each substantial experiment is organized as **Prediction → Implementation → Experiment → Analysis**.

### Use of generative AI
_TODO: describe how you used AI (e.g. Claude for checking derivations, drafting code), and point to the part (f) verifications._

In [ ]:
using Plots

---
# Problem 1 — Comparison of Root-Finding Methods

$$
f(x) = x - \ln(x) - 25, \qquad x > 0,
$$
and we approximate the larger root, using the reference value $p = 28.344430557890227$.

Derivatives used below:
$$
f'(x) = 1 - \frac{1}{x}, \qquad f''(x) = \frac{1}{x^2}.
$$

(In Julia, `log(x)` is the natural logarithm $\ln x$.)

In [ ]:
f(x) = x - log(x) - 25;     # log is the natural log ln in Julia
df(x) = 1 - 1/x;
p = 28.344430557890227;     # reference value for the larger root

@show f(28);
@show f(30);                # opposite signs, so [28, 30] brackets the root

In [ ]:
xx = LinRange(20, 35, 200)
plot(xx, f.(xx), label="x - ln(x) - 25", xlabel="x", ylabel="f(x)")
hline!([0], label="y = 0")
vline!([28, 30], label="[a, b] = [28, 30]", linestyle=:dash)
title!("f(x) near the larger root")

## 1(a) Bisection: iterations needed for error $< 10^{-8}$

Using the bound from class: after $n$ bisections of $[a_0, b_0]$, the midpoint is within $\dfrac{b_0-a_0}{2^{n+1}}$ of the root. Requiring this to be smaller than $\tau = 10^{-8}$:
$$
\frac{b_0 - a_0}{2^{n+1}} < \tau
\iff n > \log_2\!\left(\frac{b_0-a_0}{\tau}\right) - 1
= \log_2\!\left(\frac{2}{10^{-8}}\right) - 1
\approx 27.58 - 1 = 26.58 .
$$
So
$$
\boxed{n = 27}
$$

**Check:**
* $n = 27$: $\dfrac{2}{2^{28}} \approx 7.45\times10^{-9} < 10^{-8}$ ✓
* $n = 26$: $\dfrac{2}{2^{27}} \approx 1.49\times10^{-8} > 10^{-8}$ ✗

**Convention note.** Some texts label the first midpoint $p_1$ and use $|p_n - p| \le \dfrac{b-a}{2^n}$, which gives $n = 28$. Both conventions refer to the same point: the 28th midpoint, computed after 27 bisections.

In [ ]:
a = 28;
b = 30;
τ = 10^(-8);
@show log2((b - a)/τ) - 1;     # criterion from class (Oct 1): n ≥ this
@show (b - a)/2^(27 + 1);      # error bound after 27 bisections: below τ
@show (b - a)/2^(26 + 1);      # error bound after 26 bisections: above τ

So this example needs 27 bisections, which is the 28th midpoint.

## 1(b) Newton's method from $p_0 = 29$: theoretical prediction

**Convergence order.** Newton's method converges quadratically to a simple root, i.e. a root with $f'(p) \neq 0$, provided $p_0$ is sufficiently close. Here
$$
f'(p) = 1 - \frac{1}{p} \approx 1 - \frac{1}{28.34} \approx 0.965 \neq 0,
$$
so $p$ is a simple root and Newton converges **at least quadratically**:
$$
|e_{n+1}| = O\!\left(|e_n|^2\right), \qquad e_n = p_n - p .
$$
Moreover $f''(p) = 1/p^2 \neq 0$, so the convergence is **exactly quadratic** ($\alpha = 2$):
$$
\lim_{n\to\infty}\frac{|e_{n+1}|}{|e_n|^2} = \left|\frac{f''(p)}{2f'(p)}\right| \neq 0 .
$$

**Correct digits.** If $|e_n| \approx 10^{-k}$, then $|e_{n+1}| \approx C\cdot 10^{-2k}$. So once the iteration is sufficiently close to the root, **the number of correct digits roughly doubles each iteration** (about $1 \to 2 \to 4 \to 8 \to 16$), until it reaches the limit of double precision (about 16 digits).

**Checking the order from data (used in part d).** If $|e_{n+1}| \approx C|e_n|^\alpha$ and $|e_n| \approx C|e_{n-1}|^\alpha$, dividing the two and taking logs gives
$$
\alpha \approx \frac{\ln|e_{n+1}| - \ln|e_n|}{\ln|e_n| - \ln|e_{n-1}|}.
$$
The constant $C$ cancels. We predict $\alpha \approx 2$.

**Can the order alone tell us how many iterations are needed for an error $< 10^{-8}$?** No. The order only describes how each error relates to the previous one. The actual number of iterations also depends on:
1. **the starting error** $|p_0 - p|$, i.e. how many correct digits we start with;
2. **the asymptotic constant** $C = \left|\dfrac{f''(p)}{2f'(p)}\right|$, which the order does not tell us;
3. **when the asymptotic regime begins.** Quadratic convergence is only guaranteed "sufficiently close" to $p$, and the theorem does not say how close that is.

Two methods with the same order but different $C$ or $p_0$ can need very different numbers of iterations, and one of them may not converge at all.

## 1(c) Bisection on $[28, 30]$

**Prediction.** _TODO_

**Implementation.** _TODO_

**Experiment.** _TODO_

**Analysis.** _TODO_

Starting point: `bisection_method` from class (Sept 29 / Oct 1). _TODO: modify it to store $p_n$ at every iteration so we can print the table $n$, $p_n$, $|p_n - p|$, $|p_n - p_{n-1}|$, $|f(p_n)|$._

In [ ]:
function bisection_method(f, a, b; nmax=100, δ = 1e-6, ϵ = 1e-6)
    if (f(a) * f(b) > 0)
        # throws an error and prints the message, and it terminates the function
        error("The function must have opposite signs at the endpoints of the interval.")
    end

    # need to include this to ensure we are always in scope
    c = 0;

    for n in 1:nmax
        half_width = (b-a)/2;
        c = a + half_width; # compute the midpoint
        # check our result
        if abs(f(c)) < ϵ || half_width < δ
            if abs(f(c)) < ϵ
                @info("Root found at x = $c after $n iterations based on function value")
            elseif half_width < δ
                @info("Root found at x = $c after $n iterations based on interval width")
            end
            return c;
        # otherwise proceed to the next iteration
        elseif f(a)*f(c) < 0
            b = c;
        else
            a = c
        end
    end
    # prints an error message, but does not break the function
    @error("$nmax iterations reached without finding a root.")
    return c
end

In [ ]:
# TODO: run bisection on [28, 30] and print the table

## 1(d) Newton's method from $p_0 = 29$

**Prediction.** _TODO_

**Implementation.** _TODO_

**Experiment.** _TODO_

**Analysis.** _TODO_

Starting point: `newton_method` from class (Oct 8). _TODO: modify it to store every iterate for the table, and compute $\alpha$ from the formula in 1(b)._

In [ ]:
function newton_method(f, df, x0; nmax = 100, δ=1e-6, ϵ = 1e-6)
    x = x0;
    for n in 1:nmax
        fx = f(x);
        dfx = df(x);
        if abs(fx) < ϵ
            @info("Newton converged to root after $n iterations -- small magnitude of f(x)")
            return x
        end
        x_new = x - fx / dfx;
        if abs(x_new - x) < δ
            @info("Newton converged to root after $n iterations -- small change in x")
            return x_new
        end
        x = x_new;
    end
    @error("Newton did not converge after $nmax iterations")
    return x
end

In [ ]:
# TODO: run Newton from p0 = 29 and print the table

## 1(e) Newton's method from $p_0 = 1.0000000001$

**Prediction.** _TODO_

**Implementation.** _TODO_

**Experiment.** _TODO_

**Analysis.** _TODO_

In [ ]:
# TODO

## 1(f) Verifying an AI claim

**AI claim:** _TODO_

**Independent verification:** _TODO_

**Correction (if needed):** _TODO_

---
# Problem 2 — Polynomial Roots

$$
f(x) = 230x^4 + 18x^3 + 9x^2 - 221x - 9
$$

## 2(a) Plot on $[-1, 1]$ and initial guesses

**Prediction.** _TODO_

**Implementation.** _TODO_

**Experiment.** _TODO_

**Analysis.** _TODO_

In [ ]:
# TODO

## 2(b) Newton's method for the two real roots

**Prediction.** _TODO_

**Implementation.** _TODO_

**Experiment.** _TODO_

**Analysis.** _TODO_

In [ ]:
# TODO

## 2(c) Complex initial guesses

**Prediction.** _TODO_

**Implementation.** _TODO_

**Experiment.** _TODO_

**Analysis.** _TODO_

In [ ]:
# TODO

## 2(d) Iteration count vs. $p_0$ for `p0_vals = 0:0.001:1`

**Prediction.** _TODO_

**Implementation.** _TODO_

**Experiment.** _TODO_

**Analysis.** _TODO_

In [ ]:
# TODO

## 2(e) $f'(x)$ on $[0, 1]$

**Prediction.** _TODO_

**Implementation.** _TODO_

**Experiment.** _TODO_

**Analysis.** _TODO_

In [ ]:
# TODO

## 2(f) Verifying an AI claim

**AI claim:** _TODO_

**Independent verification:** _TODO_

**Correction (if needed):** _TODO_

---
# Problem 3 — Fixed-Point Iteration for $x^5 = 7$

_TODO: copy the four mappings $g_1, \dots, g_4$ exactly from the assignment PDF._

## 3(a) Fixed points and $g_i'(p)$

**Prediction.** _TODO_

**Implementation.** _TODO_

**Experiment.** _TODO_

**Analysis.** _TODO_

In [ ]:
# TODO

## 3(b) Predictions from $p_0 = 1$

**Prediction.** _TODO_

**Implementation.** _TODO_

**Experiment.** _TODO_

**Analysis.** _TODO_

In [ ]:
# TODO

## 3(c) Running the four iterations from $p_0 = 1$

**Prediction.** _TODO_

**Implementation.** _TODO_

**Experiment.** _TODO_

**Analysis.** _TODO_

In [ ]:
# TODO

## 3(d) Which mapping is a known root-finding method?

**Prediction.** _TODO_

**Implementation.** _TODO_

**Experiment.** _TODO_

**Analysis.** _TODO_

In [ ]:
# TODO

## 3(e) Dependence on the initial guess

**Prediction.** _TODO_

**Implementation.** _TODO_

**Experiment.** _TODO_

**Analysis.** _TODO_

In [ ]:
# TODO

## 3(f) Verifying an AI claim

**AI claim:** _TODO_

**Independent verification:** _TODO_

**Correction (if needed):** _TODO_